In [1]:
import pandas as pd
import matplotlib
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [ ]:
def percent_nan_per_col(df, column):
        return (df[column].isnull().sum() / len(df)) * 100



In [83]:
folder = "C:/Users/Michael/Desktop/CS-513/Final Project/"

df = pd.read_csv(folder + "flight_cases.csv", na_values="", keep_default_na=False)

total_rows = len(df)

df = df.drop(columns=["Source"])
df = df.dropna(how='all')
df = df.reset_index()

print(df.describe())
df = df.drop(columns=["DocketUrl", "ReportUrl", "RepGenFlag", "ProbableCause", "Findings", "N#", "Mkey", "SerialNumber", "OriginalPublishedDate", "DocketOriginalPublishedDate" ])
print(df.dtypes)
print(df.describe())
df.tail(5)

             index           Mkey  MinorInjuryCount    Longitude   RepGenFlag
count  8428.000000    8411.000000       7808.000000  7.053000e+03         0.0
mean   4217.709065  138673.183450          0.309298  1.472996e+02         NaN
std    2435.610995   44037.535802          2.282587  3.296572e+04         NaN
min       0.000000  100756.000000          0.000000 -1.004241e+06         NaN
25%    2108.750000  103387.500000          0.000000 -1.119164e+02         NaN
50%    4218.500000  105922.000000          0.000000 -9.266506e+01         NaN
75%    6326.250000  193528.500000          0.000000 -8.100000e+01         NaN
max    8435.000000  200059.000000        161.000000  1.140549e+06         NaN
index                     int64
NtsbNo                   object
EventType                object
EventDate                object
City                     object
State                    object
Country                  object
ReportNo                 object
HasSafetyRec             object
Mode      

,index,NtsbNo,EventType,EventDate,City,State,Country,ReportNo,HasSafetyRec,Mode,...,EngineType,Scheduled,PurposeOfFlight,FAR,AirCraftDamage,WeatherCondition,Operator,BroadPhaseofFlight,ReportStatus,MostRecentReportType
8423,8431,CEN25LA070,ACC,2025-01-02T19:54:00Z,Stringtown,Mississippi,United States,NaN,False,Aviation,...,TF,NSCH,NaN,135,Substantial,VMC,CORPORATE FLIGHT MANAGEMENT INC,Enroute,Completed,Final
8424,8432,WPR25FA072,ACC,2025-01-02T15:09:00Z,Fullerton,California,United States,NaN,False,Aviation,...,NaN,NaN,PERS,091,Substantial,VMC,NaN,Approach,In work,Prelim
8425,8433,WPR25LA080,ACC,2025-01-02T13:30:00Z,Round Mountain,Nevada,United States,NaN,False,Aviation,...,NaN,NaN,PERS,091,Substantial,VMC,NaN,Enroute,In work,Prelim
8426,8434,GAA25WA055,ACC,2025-01-02T11:15:00Z,Fassberg,Other Foreign,Germany,NaN,False,Aviation,...,NaN,NaN,NaN,NUSN,Substantial,NaN,NaN,Landing,N/A,Prelim
8427,8435,ERA25LA088,ACC,2025-01-01T02:20:00Z,Naples,Florida,United States,NaN,False,Aviation,...,NaN,NaN,PERS,091,Substantial,IMC,NaN,Takeoff,In work,Prelim


### Dropping cols with > 40% null

In [80]:

cols_to_drop = []
for i in df.columns:
    n = percent_nan_per_col(df, i)
    if (n > 40):
        print(i + ": " + f"{n:.2f}" + "\n")
        cols_to_drop.append(i)
df = df.drop(columns=cols_to_drop)
print(df.columns)
        



ReportNo: 99.75

OnGroundInjuryCount: 66.47

EventID: 99.81

AirportName: 44.76

Scheduled: 84.50

Operator: 62.55

Index(['index', 'NtsbNo', 'EventType', 'EventDate', 'City', 'State', 'Country',
       'HasSafetyRec', 'Mode', 'ReportType', 'HighestInjuryLevel',
       'FatalInjuryCount', 'SeriousInjuryCount', 'MinorInjuryCount',
       'OnboardInjuryCount', 'Latitude', 'Longitude ', 'Make', 'Model',
       'AirCraftCategory', 'AirportID', 'AmateurBuilt', 'NumberOfEngines',
       'EngineType', 'PurposeOfFlight', 'FAR', 'AirCraftDamage',
       'WeatherCondition', 'BroadPhaseofFlight', 'ReportStatus',
       'MostRecentReportType'],
      dtype='object')


### Cleaning bad records

In [ ]:
# NtsbNo

for index, row in df.iterrows():
    if ((len(row.NtsbNo) != 10) or ("." in row.NtsbNo) or (row.NtsbNo == None)):
        print(index)
        df = df.drop(index)
df = df.reset_index()

total_rows_dropped = total_rows - len(df)

# Maybe do more cleaning, but not enough time, need to focus on model prep for now.
    

115
1279
1714
2153
3438
3560
4673
5746
5765
6337
6338
6488
6501
6597
6779
7436
7495
26


In [ ]:
df.to_csv('clean_flight_cases.csv', index=False)

In [85]:
dtypes = {"NtsbNo": "str",
    "EventType": "str",
    "EventDate": "str",
    "City": "str",
    "State": "str",
    "Country": "str",
    "HasSafetyRec": "bool",
    "Mode": "str",
    "ReportType": "str",
    "HighestInjuryLevel": "object",
    "FatalInjuryCount": "float64",
    "SeriousInjuryCount": "float64",
    "MinorInjuryCount": "float64",
    "OnboardInjuryCount": "float64",
    "Latitude": "float64",
    "Longitude": "float64",
    "Make": "str",
    "Model": "str",
    "AirCraftCategory": "str",
    "AirportID": "str",
    "AmateurBuilt": "str",
    "NumberOfEngines": "str",
    "EngineType": "str",
    "PurposeOfFlight": "str",
    "FAR": "str",
    "AirCraftDamage": "str",
    "WeatherCondition": "str",
    "BroadPhaseofFlight": "str",
    "ReportStatus": "str",
    "MostRecentReportType": "str" }

df2 = pd.read_csv(folder + "clean_flight_cases.csv", na_values=["", " "], keep_default_na=False, dtype=dtypes)

In [86]:


print(df2.dtypes)

index                            int64
NtsbNo                          object
EventType                       object
EventDate                       object
City                            object
State                           object
Country                         object
HasSafetyRec                      bool
Mode                            object
ReportType                      object
OriginalPublishedDate           object
DocketOriginalPublishedDate     object
HighestInjuryLevel              object
FatalInjuryCount               float64
SeriousInjuryCount             float64
MinorInjuryCount               float64
OnboardInjuryCount             float64
Latitude                       float64
Longitude                      float64
Make                            object
Model                           object
AirCraftCategory                object
AirportID                       object
AmateurBuilt                    object
NumberOfEngines                 object
EngineType               